# Sistema de agrupación de casas

In [ ]:
import pandas as pd
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from utils2 import get_classifier_metrics
import pickle

Donde són los lugares que tengo casas mas grandes o mas pequeñas 

clasificar casas según su la región en la que se encuentren y del ingreso medio

## Analisis exploratorio de datos.

### Cargar el conjunto de datos.

In [158]:
df = pd.read_csv("../data/raw/housing.csv")
df

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422
...,...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09,0.781
20636,2.5568,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21,0.771
20637,1.7000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22,0.923
20638,1.8672,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32,0.847


> En este caso solo nos interesan las columnas ´Latitude´, `Longitude` y `MedInc`.

In [159]:
df = df[['Latitude', 'Longitude', 'MedInc']]
df

,Latitude,Longitude,MedInc
0,37.88,-122.23,8.3252
1,37.86,-122.22,8.3014
2,37.85,-122.24,7.2574
3,37.85,-122.25,5.6431
4,37.85,-122.25,3.8462
...,...,...,...
20635,39.48,-121.09,1.5603
20636,39.49,-121.21,2.5568
20637,39.43,-121.22,1.7000
20638,39.43,-121.32,1.8672


### Analisis descriptivo

In [160]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Latitude   20640 non-null  float64
 1   Longitude  20640 non-null  float64
 2   MedInc     20640 non-null  float64
dtypes: float64(3)
memory usage: 483.9 KB


#### Observaciones:
> - Existes un total de 20640 filas y 3 columnas.
> - Ninguna de las características presetan valores nulos.

### Limpieza de Datos

In [161]:
df.duplicated().sum()

np.int64(5)

In [162]:
df.drop_duplicates()

,Latitude,Longitude,MedInc
0,37.88,-122.23,8.3252
1,37.86,-122.22,8.3014
2,37.85,-122.24,7.2574
3,37.85,-122.25,5.6431
4,37.85,-122.25,3.8462
...,...,...,...
20635,39.48,-121.09,1.5603
20636,39.49,-121.21,2.5568
20637,39.43,-121.22,1.7000
20638,39.43,-121.32,1.8672


> Ahora el DataFrame está libre de duplicados


## Construye un K-Means

In [163]:
# Entrenar el modelo
model = KMeans(n_clusters = 6, random_state = 42)
model.fit(df)

predictions = model.predict(df)
predictions

array([2, 2, 2, ..., 1, 1, 1], shape=(20640,), dtype=int32)

In [164]:
df.loc[:, 'cluster'] = predictions

/tmp/ipykernel_15457/1469779687.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:, 'cluster'] = predictions


In [165]:
df

,Latitude,Longitude,MedInc,cluster
0,37.88,-122.23,8.3252,2
1,37.86,-122.22,8.3014,2
2,37.85,-122.24,7.2574,2
3,37.85,-122.25,5.6431,2
4,37.85,-122.25,3.8462,1
...,...,...,...,...
20635,39.48,-121.09,1.5603,1
20636,39.49,-121.21,2.5568,1
20637,39.43,-121.22,1.7000,1
20638,39.43,-121.32,1.8672,1


In [166]:
with open('../data/processed/housing_kmeans.pkl', 'wb') as file:
    pickle.dump(df, file)

## Split

In [167]:

X = df.drop('cluster', axis=1)
y = df['cluster']

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=18)

### Random Forest

In [168]:
rf_model = RandomForestClassifier(random_state=18)
rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [169]:
rf_model.feature_importances_

array([0.28752589, 0.24834568, 0.46412843])

In [170]:
y_pred_test = rf_model.predict(X_test)
y_pred_train = rf_model.predict(X_train)
y_pred_test, y_pred_train

(array([2, 0, 2, ..., 3, 2, 1], shape=(4128,), dtype=int32),
 array([3, 1, 4, ..., 1, 1, 4], shape=(16512,), dtype=int32))

In [171]:
get_classifier_metrics(y_pred_test, y_test, y_pred_train, y_train)

,Accuracy,F1 Score,Precision,Recall
Train set,1.000000,1.000000,1.000000,1.000000
Test set,0.995397,0.995397,0.995397,0.995397


In [173]:
with open('../data/processed/housing_rf.pkl', 'wb') as file:
    pickle.dump(df, file)